In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from tools.utils import *
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
from tools.plasticity_measures import *

In [4]:
processed_data_dir = Path('processed_data/tous')
embeddings = np.loadtxt(processed_data_dir / 'embedding.tsv')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')
print(len(embeddings), len(metadata))
# 84621 84621

233880 233880


In [7]:
metadata['contract_id'] = metadata.apply(lambda x: f"{x['platform']}_{x['year']}", axis=1)
metadata['idx'] = range(len(metadata))
metadata['position'] = metadata['idx'] - metadata.groupby('contract_id')['idx'].transform('min')
metadata.sort_values(['platform', 'year', 'position'], inplace=True)

In [ ]:

platforms = metadata['platform'].unique()

results_platform = []

for platform in tqdm(platforms):
    platform_ids = list(metadata[metadata['platform']==platform]['contract_id'].unique())
    for p1, p2 in list(zip(platform_ids[:-1], platform_ids[1:])):
        X_text = list(metadata[metadata['contract_id']==p1]['sentence'].values)
        Y_text = list(metadata[metadata['contract_id']==p2]['sentence'].values)
        X = embeddings[metadata['contract_id']==p1]
        Y = embeddings[metadata['contract_id']==p2]
        df = contract_reading_effort_density(X_text,X, Y_text, Y)
        #df = assignment_based_effort(X, Y)
        df['date'] = p2.split('_')[-1]
        results_platform.append(df)

  0%|          | 0/50 [00:00<?, ?it/s]/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
 48%|████▊     | 24/50 [1:32:46<1:29:59, 207.66s/it]

In [ ]:
df_all = pd.concat([pd.DataFrame.from_dict(d, orient='index').T for d in results_platform], ignore_index=True, axis=0)
df_all['year'] = df_all['date'].apply(lambda x: int(x[:4]))

In [ ]:
sns.lineplot(data=df_all, x='year', y='total_effort')

In [ ]:
df_all.groupby('year').agg({'total_effort': 'sum' }).plot()